In [3]:
import requests
import sys
import re
import string
import random 
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
path = tf.keras.utils.get_file('alice.txt', 'https://www.gutenberg.org/files/19033/19033-0.txt')
text = open(path, 'rb').read().decode(encoding='utf-8')

text = text[485:54815].lower()

replaced = ["[illustration]", "-", "\"", "_", "...", "__", "*", "ù"]

for r in replaced:
    text = text.replace(r, "")

text = text.strip()


In [5]:
uniq_chars = sorted(set(text))
vocab_size = len(uniq_chars)

print(f"Printing {len(uniq_chars)} characters: {uniq_chars}")


Printing 40 characters: ['\n', '\r', ' ', '!', "'", '(', ')', ',', '.', ':', ';', '?', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
char2int = {u:i for i, u in enumerate(uniq_chars)}
int2char = np.array(uniq_chars)

In [7]:
seq_length = 100
step = 3
sentences = []
next_chars = []

for i in range(0, len(text) - seq_length, step):
    sentences.append(text[i : i + seq_length])
    next_chars.append(text[i + seq_length])

print(f"Total training sequences: {len(sentences)}")

Total training sequences: 17625


In [8]:
X = np.zeros((len(sentences), seq_length), dtype=np.float32)
y = np.zeros(len(sentences), dtype=np.float32)

for i, sentence in enumerate(sentences):
    for j, ch in enumerate(sentence):
        X[i, j] = char2int[ch]
    y[i] = char2int[next_chars[i]]

In [9]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 128, input_length=seq_length),
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

d:\Programming\my_python\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

In [13]:
# early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Training the Baby Gpt!")

# training = model.fit(X, y, batch_size=128, epochs=75, validation_split=0.1, callbacks=[early_stop])
training = model.fit(X, y, batch_size=128, epochs=25)

Training the Baby Gpt!
Epoch 1/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 55s 400ms/step - accuracy: 0.1845 - loss: 2.9805
Epoch 2/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 61s 445ms/step - accuracy: 0.2945 - loss: 2.5038
Epoch 3/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 31s 224ms/step - accuracy: 0.3447 - loss: 2.2973
Epoch 4/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 30s 216ms/step - accuracy: 0.3722 - loss: 2.1847
Epoch 5/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 30s 215ms/step - accuracy: 0.3936 - loss: 2.1105
Epoch 6/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 30s 214ms/step - accuracy: 0.4144 - loss: 2.0386
Epoch 7/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 29s 213ms/step - accuracy: 0.4315 - loss: 1.9740
Epoch 8/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 29s 213ms/step - accuracy: 0.4401 - loss: 1.9244
Epoch 9/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 30s 218ms/step - accuracy: 0.4549 - loss: 1.8730
Epoch 10/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 30s 218ms/step - accuracy: 0.4693 - loss: 1.8248
Epoch 11/25
138/138 ━━━━━━━━━━━━━━━━━━━━ 30s 216ms/step - accuracy: 0.4767 - loss: 1.7

In [16]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(length=400, temperature=0.5):
    start_index = random.randint(0, len(text) - seq_length - 1)
    seed_text = text[start_index : start_index + seq_length]
    
    print(f"--- Seed: \"{seed_text}\"")
    print("--- Generated Text: ", end="")

    for i in range(length):
        x_pred = np.zeros((1, seq_length))
        for t, char in enumerate(seed_text):
            x_pred[0, t] = char2int[char]

        preds = model.predict(x_pred, verbose=0)[0]
        
        next_index = sample(preds, temperature)
        next_char = int2char[next_index]

        seed_text = seed_text[1:] + next_char

        sys.stdout.write(next_char)
        sys.stdout.flush()
    print()

generate_text(temperature=0.3)

--- Seed: "n, but those serpents! there's no pleasing
them!

alice was more and more puzzled.

as if it wa"
--- Generated Text: s a thould and the same the poor.

alice was was as she said the cound and and was the great was door and was a little had fourther the white the rabbit was a srow to stepbled to the had fine to the the same she was a said to keate and she head the poor and for the nour was the cound it the goor.

the taid the white thinking her her here the had foor and the white was the rabbit was at the whi
